# Time-window correlation matrices (single patient)

Compute a few correlation matrices across time windows for one band.

**Legacy notebooks merged:**
- TEST_per_patient_time_windows.ipynb
- tests.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
from lrgsglib.utils.basic.signals import bandpass_sos

patient = list_patients(Path('data/stereoeeg_patients'))[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

recording = load_patient_dataset_robust(patient, Path('data/stereoeeg_patients'), phases=[phase])[phase]
timeseries = recording.timeseries
fs = float(recording.parameters.get('fs'))

window_sec = 10.0
overlap = 0.25
window_len = int(window_sec * fs)
step = int(window_len * (1.0 - overlap))

n_samples = timeseries.shape[1]
indices = list(range(0, n_samples - window_len + 1, step))
indices = indices[:6]

corr_windows = []
for start in indices:
    window = timeseries[:, start:start + window_len]
    low, high = BRAIN_BANDS[band]
    filtered = bandpass_sos(window, low, high, fs, 4)
    corr = build_corr_network(filtered, filter_type='abs', zero_diagonal=True)
    corr_windows.append(corr)

len(corr_windows)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(corr_windows), figsize=(4 * len(corr_windows), 4))
if len(corr_windows) == 1:
    axes = [axes]
for ax, corr in zip(axes, corr_windows):
    im = ax.imshow(corr, cmap='viridis', vmin=0, vmax=1)
    ax.set_title('corr window')
fig.colorbar(im, ax=axes, shrink=0.6)
plt.tight_layout()
plt.show()